# Robustness evaluation and controlled rerun

This notebook executes the post-Task-3 milestone. It evaluates existing Stage A and RINE checkpoints, retrains controlled RINE across seeds 42/43/44, then tests retained frequency and Lab fusion candidates. It never reads `final_test`; Tasks 9 and 10 remain downstream.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT = Path('/content/cya-techjam26')
ARTIFACTS = PROJECT / 'artifacts'
ROBUSTNESS = ARTIFACTS / 'robustness'
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
SEEDS = (42, 43, 44)
assert PROJECT.is_dir(), PROJECT
print({'project': str(PROJECT), 'robustness': str(ROBUSTNESS), 'prior': str(PRIOR)})

## 1. Preflight
Run after mounting Drive, installing with `make install-colab`, and staging the fixed-Q96 manifest and images under `/content`.

In [ ]:
subprocess.run(['make', 'smoke'], cwd=PROJECT, check=True)
subprocess.run(['make', 'robustness-test'], cwd=PROJECT, check=True)

## 2. Prepare independent robustness views
This writes a development-only clean manifest, materializes all 14 Task 3 cells directly from matched-clean parents, validates the complete bank, and creates a combined feature-extraction manifest.

In [ ]:
subprocess.run(
    ['make', 'robustness-prepare', 'ARTIFACT_ROOT=/content/cya-techjam26/artifacts'],
    cwd=PROJECT, check=True,
)

## 3. Evaluate existing clean-trained checkpoints
These runs update no weights and establish the pre-retraining robustness baseline.

In [ ]:
for seed in SEEDS:
    stage_a = PRIOR / 'task4/fixed_q96' / f'seed_{seed}' / 'best_clean.pt'
    rine = PRIOR / 'task6/fixed_q96' / f'seed_{seed}' / 'best_clean.pt'
    assert stage_a.is_file(), stage_a
    assert rine.is_file(), rine
    subprocess.run(['make', 'robustness-stage-a-evaluate', f'ROBUSTNESS_SEED={seed}', f'STAGE_A_CHECKPOINT={stage_a}'], cwd=PROJECT, check=True)
    subprocess.run(['make', 'robustness-rine-evaluate', f'ROBUSTNESS_SEED={seed}', f'RINE_CHECKPOINT={rine}'], cwd=PROJECT, check=True)

## 4. Controlled RINE rerun
The CLIP tower stays frozen. Only the RINE layer weighting and binary head train under the balanced clean-or-one-transform sampler.

In [ ]:
for seed in SEEDS:
    subprocess.run(['make', 'robustness-rine-train', f'ROBUSTNESS_SEED={seed}'], cwd=PROJECT, check=True)

## 5. Extract retained auxiliary features
Frequency extraction excludes phase during fusion. Auxiliary extraction supplies Lab; RGB, PRNU, and optics are not fused by this milestone.

In [ ]:
subprocess.run(['make', 'robustness-frequency-extract'], cwd=PROJECT, check=True)
subprocess.run(['make', 'robustness-lab-extract'], cwd=PROJECT, check=True)

## 6. Train individual fusion candidates
Each candidate freezes its controlled-RINE parent and trains only the auxiliary projection and fusion head.

In [ ]:
for variant in ('frequency', 'lab'):
    for seed in SEEDS:
        subprocess.run(['make', 'robustness-fusion-train', f'ROBUSTNESS_FUSION_VARIANT={variant}', f'ROBUSTNESS_SEED={seed}'], cwd=PROJECT, check=True)

## 7. Apply retention gates
A candidate is retained only if its mean 50/50 score strictly improves and neither mean class accuracy regresses by more than one percentage point.

In [ ]:
decisions = {}
for variant in ('frequency', 'lab'):
    output = ROBUSTNESS / 'reports' / variant
    subprocess.run([
        'python', 'scripts/compare_robustness_candidate.py',
        '--parent-root', str(ROBUSTNESS / 'train-controlled-rine'),
        '--candidate-root', str(ROBUSTNESS / f'rine_{variant}'),
        '--candidate-name', f'rine_{variant}',
        '--output', str(output),
    ], cwd=PROJECT, check=True)
    decisions[variant] = json.loads((output / 'retention_decision.json').read_text())['decision']
decisions

## 8. Conditional combined fusion
Run the combined candidate only if both individual additions passed. Otherwise controlled RINE is the pre-Task-9 handoff.

In [ ]:
if decisions == {'frequency': 'retain', 'lab': 'retain'}:
    for seed in SEEDS:
        subprocess.run(['make', 'robustness-fusion-train', 'ROBUSTNESS_FUSION_VARIANT=frequency_lab', f'ROBUSTNESS_SEED={seed}'], cwd=PROJECT, check=True)
    combined_output = ROBUSTNESS / 'reports' / 'frequency_lab'
    subprocess.run([
        'python', 'scripts/compare_robustness_candidate.py',
        '--parent-root', str(ROBUSTNESS / 'train-controlled-rine'),
        '--candidate-root', str(ROBUSTNESS / 'rine_frequency_lab'),
        '--candidate-name', 'rine_frequency_lab',
        '--output', str(combined_output),
    ], cwd=PROJECT, check=True)
    print(json.loads((combined_output / 'retention_decision.json').read_text())['decision'])
else:
    print('Combined candidate skipped:', decisions)

## 9. Durable sync
After reviewing completion markers and reports, copy `/content/cya-techjam26/artifacts/robustness` to `/content/drive/MyDrive/cya-techjam26/artifacts/robustness`. Do not move or delete the local copy during the run.